# PaddleOCR 한국어 파인튜닝

운송장 OCR 모델을 한국어에 맞게 파인튜닝합니다.

## 1. 환경 설정

In [20]:
# GPU 설정 (가장 먼저 실행)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '7'  # 사용할 GPU 번호
os.environ['FLAGS_fraction_of_gpu_memory_to_use'] = '0.9'

In [21]:
# 필수 패키지 설치
!pip install paddlepaddle-gpu==2.6.1.post120 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q
!pip install paddleocr faker tqdm PyYAML opencv-python -q

In [22]:
# GPU 확인
import paddle
print(f'PaddlePaddle 버전: {paddle.__version__}')
print(f'GPU 사용 가능: {paddle.is_compiled_with_cuda()}')
print(f'GPU 개수: {paddle.device.cuda.device_count()}')

if paddle.is_compiled_with_cuda():
    paddle.device.set_device('gpu:0'
                            )
    x = paddle.randn([2, 3])
    print(f'GPU 테스트 성공: {x.place}')

PaddlePaddle 버전: 2.6.1
GPU 사용 가능: True
GPU 개수: 1
GPU 테스트 성공: Place(gpu:0)


## 2. 데이터 생성

In [4]:
# 데이터 생성 (3만개)
# !python generate_ocr_data.py -n 30000 --clear --effect "none:40,blur:20,mosaic:15,noise:15,combined:10" --strength 8 --random-strength

## 3. PaddleOCR 설치 및 데이터 전처리

In [23]:
import json
import shutil
import random
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# 경로 설정
BASE_DIR = Path('.').resolve()
GENERATED_DIR = BASE_DIR / 'generated'
DATA_DIR = BASE_DIR / 'paddle_data'
PADDLEOCR_DIR = BASE_DIR / 'PaddleOCR'

In [24]:
# PaddleOCR 클론 (없으면)
if not PADDLEOCR_DIR.exists():
    !git clone https://github.com/PaddlePaddle/PaddleOCR.git
    !pip install -r PaddleOCR/requirements.txt -q
else:
    print('PaddleOCR이 이미 존재합니다.')

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 312043, done.
remote: Counting objects: 100% (1285/1285), done.
remote: Compressing objects: 100% (313/313), done.
remote: Total 312043 (delta 1173), reused 973 (delta 972), pack-reused 310758 (from 3)
Receiving objects: 100% (312043/312043), 1.65 GiB | 20.34 MiB/s, done.
Resolving deltas: 100% (246946/246946), done.


In [25]:
# 데이터 전처리: PaddleOCR 형식으로 변환
def convert_to_paddle_format(generated_dir, output_dir, train_ratio=0.9):
    train_img_dir = output_dir / 'train' / 'images'
    val_img_dir = output_dir / 'val' / 'images'
    
    if output_dir.exists():
        shutil.rmtree(output_dir)
    
    train_img_dir.mkdir(parents=True, exist_ok=True)
    val_img_dir.mkdir(parents=True, exist_ok=True)
    
    # 라벨 로드
    labels_file = generated_dir / 'labels' / 'labels.json'
    with open(labels_file, 'r', encoding='utf-8') as f:
        all_labels = json.load(f)
    
    print(f'총 {len(all_labels)}개 이미지 로드됨')
    random.shuffle(all_labels)
    
    split_idx = int(len(all_labels) * train_ratio)
    train_labels = all_labels[:split_idx]
    val_labels = all_labels[split_idx:]
    
    train_records = []
    val_records = []
    
    def process_labels(labels, img_dir, records, prefix):
        for idx, label_data in enumerate(tqdm(labels, desc=prefix)):
            img_path = generated_dir / label_data['image_path']
            if not img_path.exists():
                continue
            
            img = Image.open(img_path)
            
            for field in label_data['fields']:
                bbox = field['bbox']
                text = field['text']
                field_name = field['field_name']
                
                cropped = img.crop((bbox[0], bbox[1], bbox[2], bbox[3]))
                crop_filename = f"{prefix}_{idx:05d}_{field_name}.jpg"
                crop_path = img_dir / crop_filename
                cropped.save(crop_path, 'JPEG', quality=95)
                
                relative_path = f"images/{crop_filename}"
                records.append(f"{relative_path}\t{text}")
    
    process_labels(train_labels, train_img_dir, train_records, 'train')
    process_labels(val_labels, val_img_dir, val_records, 'val')
    
    with open(output_dir / 'train' / 'label.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(train_records))
    
    with open(output_dir / 'val' / 'label.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(val_records))
    
    print(f'Train: {len(train_records)}개, Val: {len(val_records)}개')
    return len(train_records), len(val_records)

convert_to_paddle_format(GENERATED_DIR, DATA_DIR)

FileNotFoundError: [Errno 2] No such file or directory: '/home/j-i14a403/S14P11A403/ai/PaddleOCR/generated/labels/labels.json'

In [19]:
# 한글 문자 사전 생성
def create_korean_dict(data_dir, output_file):
    chars = set()
    
    label_file = data_dir / 'train' / 'label.txt'
    with open(label_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                chars.update(parts[1])
    
    # 한글 완성형 (가-힣)
    for code in range(0xAC00, 0xD7A4):
        chars.add(chr(code))
    
    # 한글 자모 (ㄱ-ㅎ, ㅏ-ㅣ)
    for code in range(0x3131, 0x3164):
        chars.add(chr(code))
    
    # 숫자, 영문, 특수문자
    chars.update('0123456789')
    # chars.update('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')
    # chars.update(' .-_,()[]{}:;/\\@#$%&*+=<>?!"\'~`|^')
    chars.update(' -(),') 
    sorted_chars = sorted(chars)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for char in sorted_chars:
            f.write(char + '\n')
    
    print(f'문자 사전: {len(sorted_chars)}개 문자')
    return sorted_chars

create_korean_dict(DATA_DIR, DATA_DIR / 'korean_dict.txt')

FileNotFoundError: [Errno 2] No such file or directory: '/home/j-i14a403/S14P11A403/ai/paddle_data/train/label.txt'

## 4. 사전학습 모델 다운로드

In [9]:
import urllib.request
import tarfile

pretrain_dir = PADDLEOCR_DIR / 'pretrain_models'
pretrain_dir.mkdir(parents=True, exist_ok=True)

model_name = "korean_PP-OCRv3_rec_train"
model_url = f"https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/{model_name}.tar"
model_tar = pretrain_dir / f"{model_name}.tar"
model_dir = pretrain_dir / model_name

if not model_dir.exists():
    print(f'다운로드 중: {model_url}')
    urllib.request.urlretrieve(model_url, model_tar)
    
    print('압축 해제 중...')
    with tarfile.open(model_tar, 'r') as tar:
        tar.extractall(pretrain_dir)
    
    model_tar.unlink()
    print(f'완료: {model_dir}')
else:
    print(f'이미 존재: {model_dir}')

PRETRAIN_MODEL_PATH = str(model_dir / 'best_accuracy')

이미 존재: /home/j-i14a403/S14P11A403/ai/PaddleOCR/pretrain_models/korean_PP-OCRv3_rec_train


## 5. 학습 설정 파일 생성

In [10]:
# 학습 파라미터
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.0005
NUM_WORKERS = 8

In [16]:
config_content = f'''Global:
  debug: false
  use_gpu: true
  epoch_num: {EPOCHS}
  log_smooth_window: 20
  print_batch_step: 50
  save_model_dir: ./output/rec_korean_finetune
  save_epoch_step: 5
  eval_batch_step: [0, 1000]
  cal_metric_during_train: true
  pretrained_model: {PRETRAIN_MODEL_PATH}
  checkpoints:
  save_inference_dir:
  use_visualdl: false
  infer_img:
  character_dict_path: {str(DATA_DIR / 'korean_dict.txt')}
  max_text_length: 50
  infer_mode: false
  use_space_char: true
  distributed: false
  save_res_path: ./output/rec/predicts.txt

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: {LEARNING_RATE}
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 1.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: MobileNetV1Enhance
    scale: 0.5
    last_conv_stride: [1, 2]
    last_pool_type: avg
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 64
            depth: 2
            hidden_dims: 120
            use_guide: True
          Head:
            fc_decay: 0.00001
      - SARHead:
          enc_dim: 512
          max_text_length: 50

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc
  ignore_space: False

Train:
  dataset:
    name: SimpleDataSet
    data_dir: {str(DATA_DIR / 'train')}
    ext_op_transform_idx: 1
    label_file_list:
      - {str(DATA_DIR / 'train' / 'label.txt')}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecConAug:
          prob: 0.5
          ext_data_num: 2
          image_shape: [48, 320, 3]
      - RecAug:
      - MultiLabelEncode:
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: true
    batch_size_per_card: {BATCH_SIZE}
    drop_last: true
    num_workers: {NUM_WORKERS}
    use_shared_memory: false

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: {str(DATA_DIR / 'val')}
    label_file_list:
      - {str(DATA_DIR / 'val' / 'label.txt')}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: {BATCH_SIZE}
    num_workers: {NUM_WORKERS}
    use_shared_memory: false
'''

config_file = DATA_DIR / 'rec_korean_finetune.yml'
with open(config_file, 'w', encoding='utf-8') as f:
    f.write(config_content)

print(f'설정 파일 저장: {config_file}')

설정 파일 저장: /home/j-i14a403/S14P11A403/ai/paddle_data/rec_korean_finetune.yml


## 6. 학습 실행

In [17]:
import os                                                                                        
import sys                                                                                       
                                                                                               
print("=== 커널 환경 정보 ===")                                                                  
print(f"Python 경로: {sys.executable}")                                                          
print(f"Python 버전: {sys.version}")                                                             
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', '설정 안됨')}")            
print(f"LD_LIBRARY_PATH: {os.environ.get('LD_LIBRARY_PATH', '설정 안됨')[:100]}...")             
                                                                                               
# PaddlePaddle 정보                                                                              
try:                                                                                             
  import paddle                                                                                
  print(f"\n=== PaddlePaddle 정보 ===")                                                        
  print(f"버전: {paddle.__version__}")                                                         
  print(f"CUDA 컴파일: {paddle.is_compiled_with_cuda()}")                                      
  print(f"cuDNN 버전: {paddle.version.cudnn()}")                                               
  print(f"CUDA 버전: {paddle.version.cuda()}")                                                 
except Exception as e:                                                                           
  print(f"PaddlePaddle 로드 실패: {e}")     

=== 커널 환경 정보 ===
Python 경로: /home/j-i14a403/.conda/envs/paddleocr/bin/python
Python 버전: 3.10.19 | packaged by conda-forge | (main, Jan 26 2026, 23:45:08) [GCC 14.3.0]
CUDA_VISIBLE_DEVICES: 7
LD_LIBRARY_PATH: /home/j-i14a403/.conda/envs/paddleocr/lib/python3.10/site-packages/cv2/../../lib64:/home/j-i14a403/....

=== PaddlePaddle 정보 ===
버전: 2.6.1
CUDA 컴파일: True
cuDNN 버전: 8.9.1
CUDA 버전: 12.0


In [18]:
%cd {PADDLEOCR_DIR}
!python tools/train.py -c {config_file}

/home/j-i14a403/S14P11A403/ai/PaddleOCR
Skipping import of the encryption module.
[2026/01/29 10:52:54] ppocr INFO: Architecture : 
[2026/01/29 10:52:54] ppocr INFO:     Backbone : 
[2026/01/29 10:52:54] ppocr INFO:         last_conv_stride : [1, 2]
[2026/01/29 10:52:54] ppocr INFO:         last_pool_type : avg
[2026/01/29 10:52:54] ppocr INFO:         name : MobileNetV1Enhance
[2026/01/29 10:52:54] ppocr INFO:         scale : 0.5
[2026/01/29 10:52:54] ppocr INFO:     Head : 
[2026/01/29 10:52:54] ppocr INFO:         head_list : 
[2026/01/29 10:52:54] ppocr INFO:             CTCHead : 
[2026/01/29 10:52:54] ppocr INFO:                 Head : 
[2026/01/29 10:52:54] ppocr INFO:                     fc_decay : 1e-05
[2026/01/29 10:52:54] ppocr INFO:                 Neck : 
[2026/01/29 10:52:54] ppocr INFO:                     depth : 2
[2026/01/29 10:52:54] ppocr INFO:                     dims : 64
[2026/01/29 10:52:54] ppocr INFO:                     hidden_dims : 120
[2026/01/29 10:52:54

## 7. 모델 평가

In [ ]:
!python tools/eval.py -c {config_file} -o Global.checkpoints=./output/rec_korean_finetune/best_accuracy

## 8. 추론 모델 내보내기

In [ ]:
!python tools/export_model.py -c {config_file} \
    -o Global.pretrained_model=./output/rec_korean_finetune/best_accuracy \
    Global.save_inference_dir=./output/rec_korean_finetune/inference

## 9. 최종 모델 저장

In [ ]:
# 최종 모델 복사
final_model_dir = BASE_DIR / 'models' / 'paddleocr_korean_finetuned'
final_model_dir.mkdir(parents=True, exist_ok=True)

inference_dir = PADDLEOCR_DIR / 'output' / 'rec_korean_finetune' / 'inference'

if inference_dir.exists():
    for file in inference_dir.glob('*'):
        shutil.copy(file, final_model_dir)
    
    # 문자 사전 복사
    shutil.copy(DATA_DIR / 'korean_dict.txt', final_model_dir / 'korean_dict.txt')
    
    print(f'모델 저장 완료: {final_model_dir}')
else:
    print('추론 모델이 없습니다. 먼저 내보내기를 실행하세요.')

## 10. 모델 사용법

In [ ]:
from paddleocr import PaddleOCR

# 파인튜닝된 모델 로드
ocr = PaddleOCR(
    rec_model_dir=str(final_model_dir),
    rec_char_dict_path=str(final_model_dir / 'korean_dict.txt'),
    use_angle_cls=False,
    lang='korean'
)

# 테스트 (generated/images에서 이미지 선택)
test_image = str(GENERATED_DIR / 'images' / '00001.jpg')
result = ocr.ocr(test_image, cls=False)

for line in result[0]:
    bbox, (text, confidence) = line
    print(f'텍스트: {text}, 신뢰도: {confidence:.4f}')